In [7]:
"""
Classical Machine Learning Models for Fake News Detection
Trains KNN and Naive Bayes on pre-computed TF-IDF features
"""

import numpy as np
import os
from pathlib import Path
from joblib import load, dump
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

BASE_DIR = Path(os.getcwd())
while BASE_DIR.name != "nlp-fake-news-detector-transformers" and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent

FEATURES_DIR = BASE_DIR / "data" / "saved_features"
MODELS_DIR = BASE_DIR / "results" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)


class ClassicalModels:
    """Train and evaluate KNN and Naive Bayes classifiers"""
    
    def __init__(self, X_train, X_test, y_train, y_test):
        """Initialize with training and test data"""
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        
        self.knn_model = None
        self.nb_model = None
    
    def train_knn(self, n_neighbors: int = 5):
        """Train KNN classifier"""
        self.knn_model = KNeighborsClassifier(n_neighbors=n_neighbors)
        self.knn_model.fit(self.X_train, self.y_train)

        y_pred = self.knn_model.predict(self.X_test)
        self.evaluate(self.knn_model, "KNN")

        return y_pred
    
    def train_naive_bayes(self, alpha: float = 1.0):
        """Train Naive Bayes classifier"""
        self.nb_model = MultinomialNB(alpha=alpha)
        self.nb_model.fit(self.X_train, self.y_train)

        y_pred = self.nb_model.predict(self.X_test)
        self.evaluate(self.nb_model, "Naive Bayes")

        return y_pred
    
    def evaluate(self, model, model_name: str):
        """Evaluate model and print metrics"""
        y_pred = model.predict(self.X_test)
        
        accuracy = accuracy_score(self.y_test, y_pred)
        precision = precision_score(self.y_test, y_pred)
        recall = recall_score(self.y_test, y_pred)
        f1 = f1_score(self.y_test, y_pred)
        
        print(f"\n{model_name} Results:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(self.y_test, y_pred))
        print("\nClassification Report:")
        print(classification_report(self.y_test, y_pred))
    
    def save_model(self, model, filename: str):
        """Save trained model"""
        dump(model, MODELS_DIR / filename)
        print(f"Model saved as {filename}")

## Load Pre-computed TF-IDF Features

In [ ]:
print("Loading pre-computed TF-IDF features...")
X_train_tfidf = load(FEATURES_DIR / "X_train_tfidf.joblib")
X_test_tfidf = load(FEATURES_DIR / "X_test_tfidf.joblib")
y_train = load(FEATURES_DIR / "y_train.joblib")
y_test = load(FEATURES_DIR / "y_test.joblib")

print("Features loaded successfully!")
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"X_test_tfidf shape: {X_test_tfidf.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Loading pre-computed TF-IDF features...
Features loaded successfully!
X_train_tfidf shape: (1185752, 10000)
X_test_tfidf shape: (296438, 10000)
y_train shape: (1185752,)
y_test shape: (296438,)


## Initialize and Train Models

In [9]:
# Initialize ClassicalModels with the processed TF-IDF features
print("Initializing ClassicalModels with processed data...")
models = ClassicalModels(X_train_tfidf, X_test_tfidf, y_train, y_test)

print("Models initialized successfully!")
print(f"X_train shape: {models.X_train.shape}")
print(f"X_test shape: {models.X_test.shape}")
print(f"y_train shape: {models.y_train.shape}")
print(f"y_test shape: {models.y_test.shape}")

Initializing ClassicalModels with processed data...
Models initialized successfully!
X_train shape: (1185752, 10000)
X_test shape: (296438, 10000)
y_train shape: (1185752,)
y_test shape: (296438,)


### Train KNN Classifier

In [10]:
# Train KNN classifier
print("Training KNN classifier...")
knn_predictions = models.train_knn(n_neighbors=5)

Training KNN classifier...

KNN Results:
Accuracy: 0.6680
Precision: 0.6395
Recall: 0.7523
F1-Score: 0.6914

Confusion Matrix:
[[ 87810  62123]
 [ 36287 110218]]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.59      0.64    149933
           1       0.64      0.75      0.69    146505

    accuracy                           0.67    296438
   macro avg       0.67      0.67      0.67    296438
weighted avg       0.67      0.67      0.67    296438



### Train Naive Bayes Classifier

In [11]:
# Train Naive Bayes classifier
print("Training Naive Bayes classifier...")
nb_predictions = models.train_naive_bayes(alpha=1.0)

Training Naive Bayes classifier...

Naive Bayes Results:
Accuracy: 0.7641
Precision: 0.7649
Recall: 0.7547
F1-Score: 0.7598

Confusion Matrix:
[[115943  33990]
 [ 35933 110572]]

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.77      0.77    149933
           1       0.76      0.75      0.76    146505

    accuracy                           0.76    296438
   macro avg       0.76      0.76      0.76    296438
weighted avg       0.76      0.76      0.76    296438



### Save Trained Models

In [12]:
# Save trained models
models.save_model(models.knn_model, "knn_model.joblib")
models.save_model(models.nb_model, "naive_bayes_model.joblib")

print("\nAll models trained and saved successfully!")

Model saved as knn_model.joblib
Model saved as naive_bayes_model.joblib

All models trained and saved successfully!
